# System D — Cross-attention vs Self-attention
IEMOCAP 4-class, leave-one-session-out (speaker-independent).

**Before running:** Runtime → Change runtime type → **T4 GPU**

Upload the `colab_bundle` folder to your Google Drive first (it contains
`iemocap_dataset.npz` plus the code).


In [ ]:
# 1. GPU check  (deliberately does NOT import tensorflow --
#    TF must not be imported before TF_USE_LEGACY_KERAS is set in cell 3)
!nvidia-smi -L


In [ ]:
# 2. Mount Drive and locate the bundle automatically
from google.colab import drive
import os, glob
drive.mount('/content/drive')

hits = glob.glob('/content/drive/MyDrive/**/iemocap_dataset.npz', recursive=True)
assert hits, "Could not find iemocap_dataset.npz anywhere in your Drive. Check the upload finished."
BUNDLE = os.path.dirname(hits[0])
print("found bundle:", BUNDLE)
os.chdir(BUNDLE)
!ls -la


In [ ]:
# 3. Force Keras 2 for tf.keras (TIM-Net is Keras-2 code; Colab ships Keras 3)
!pip -q install tf-keras natsort
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"     # must be set BEFORE importing tensorflow

import tensorflow as tf
# Check the MODULE IDENTITY, not a version attribute:
#   - `import keras` is always Keras 3 even when this is working
#   - tf_keras.api._v2.keras has no __version__ attribute
print("TF", tf.__version__, "| tf.keras ->", tf.keras.__name__)
assert "tf_keras" in tf.keras.__name__, (
    "tf.keras is NOT redirected - Runtime > Restart session, run cell 2, then this cell")
print("Keras 2 active. GPU:", tf.config.list_physical_devices('GPU'))

import numpy as np
z = np.load('iemocap_dataset.npz', allow_pickle=True)
print({k: z[k].shape for k in ['mfcc','gfcc','emotion','session']})


## Smoke test — one fold, few epochs
Confirms everything runs on GPU before you commit to the full matrix.

In [ ]:
!TF_USE_LEGACY_KERAS=1 python train.py --fusion cross --features mfcc+gfcc --epochs 5 --folds 5 --patience 99

## Full experiment matrix

4 configurations x 5 folds. This is the actual result table for your report.

| | MFCC | MFCC+GFCC |
|---|---|---|
| **cross** | run 1 | run 2 |
| **self**  | run 3 | run 4 |


In [ ]:
# Baselines first (fast, and they anchor the comparison)
!TF_USE_LEGACY_KERAS=1 python train.py --fusion audio_only --features mfcc      --epochs 150
!TF_USE_LEGACY_KERAS=1 python train.py --fusion text_only  --features mfcc      --epochs 150
!TF_USE_LEGACY_KERAS=1 python train.py --fusion concat     --features mfcc+gfcc --epochs 150

In [ ]:
# THE experiment: does GFCC change the cross-vs-self conclusion?
for fusion in ['cross', 'self']:
    for feats in ['mfcc', 'mfcc+gfcc']:
        print(f"\n{'='*60}\n{fusion} / {feats}\n{'='*60}")
        !TF_USE_LEGACY_KERAS=1 python train.py --fusion {fusion} --features {feats} --epochs 150

## Repeat over seeds
Run-to-run variance is ~2.5 points, so a single seed is not a result.
Report **mean ± std over 5 seeds**.

In [ ]:
for seed in range(5):
    for fusion in ['cross', 'self']:
        !TF_USE_LEGACY_KERAS=1 python train.py --fusion {fusion} --features mfcc+gfcc --epochs 150 --seed {seed}

In [ ]:
# 4. Collect all results into a table
import json, glob, numpy as np, collections
rows = collections.defaultdict(list)
for f in sorted(glob.glob('results/*.json')):
    r = json.load(open(f))
    rows[(r['fusion'], r['features'])].append((r['wa_mean'], r['ua_mean']))
print(f"{'fusion':12} {'features':12} {'WA':>16} {'UA':>16}  n")
for (fu, fe), v in sorted(rows.items()):
    wa = np.array([x[0] for x in v]) * 100
    ua = np.array([x[1] for x in v]) * 100
    print(f"{fu:12} {fe:12} {wa.mean():7.2f} +/-{wa.std():4.2f} {ua.mean():7.2f} +/-{ua.std():4.2f}  {len(v)}")